# Semana 8 — Transformada Cuántica de Fourier (QFT)
## El corazón de Shor, QPE y muchos algoritmos cuánticos

**Objetivo:** Implementar la QFT desde cero, entender su estructura recursiva y comparar su eficiencia exponencial respecto a la FFT clásica.

**Hito verificable:** Implementas la QFT para n qubits, verificas que coincide con la DFT clásica en las amplitudes, y construyes el circuito en Qiskit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
print('Librerías importadas correctamente')

## Ejercicio 8.1 — La DFT clásica y su versión cuántica
La QFT aplica la transformada de Fourier discreta a las amplitudes del estado cuántico.

In [ ]:
def dft_matrix(N):
    """Construye la matriz DFT de tamaño N×N."""
    omega = np.exp(2j * np.pi / N)
    j, k = np.meshgrid(np.arange(N), np.arange(N))
    return omega**(j*k) / np.sqrt(N)

def qft_matrix(n_qubits):
    """La QFT es la DFT con N=2^n."""
    return dft_matrix(2**n_qubits)

n = 3
QFT = qft_matrix(n)

print(f'Matriz QFT para n={n} qubits (N=8):')
print(np.round(np.abs(QFT), 3))
print()
print(f'QFT es unitaria: {np.allclose(QFT.conj().T @ QFT, np.eye(2**n))}')
print()
print('Comparación QFT vs numpy.fft:')
psi_test = np.array([1, 0, 0, 0, 0, 0, 0, 0], dtype=complex)  # |000⟩
qft_resultado = QFT @ psi_test
fft_resultado = np.fft.fft(psi_test) / np.sqrt(2**n)
print(f'QFT|000⟩ = {np.round(qft_resultado, 4)}')
print(f'FFT/√N  = {np.round(fft_resultado, 4)}')
print(f'¿Son iguales? {np.allclose(qft_resultado, fft_resultado)}')

## Ejercicio 8.2 — Estructura recursiva de la QFT
La QFT se implementa con O(n²) puertas gracias a su estructura recursiva.

In [ ]:
def Rk(k):
    """Puerta de fase controlada: añade fase e^(2πi/2^k)."""
    phase = np.exp(2j * np.pi / 2**k)
    return np.array([[1, 0], [0, phase]], dtype=complex)

def qft_matrix_recursiva(n):
    """Construye la QFT matricialmente usando la definición recursiva."""
    if n == 1:
        return np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)  # H
    
    N = 2**n
    QFT_n = np.zeros((N, N), dtype=complex)
    
    # Implementación directa: QFT[j,k] = e^(2πijk/N)/√N
    for j in range(N):
        for k in range(N):
            QFT_n[j, k] = np.exp(2j * np.pi * j * k / N) / np.sqrt(N)
    
    return QFT_n

# Verificar para varios n
for n_test in [1, 2, 3, 4]:
    Q1 = qft_matrix(n_test)
    Q2 = qft_matrix_recursiva(n_test)
    print(f'n={n_test}: QFT matricial == recursiva: {np.allclose(Q1, Q2)}')

print()
print('Complejidad:')
for n_comp in [3, 5, 10, 20, 50]:
    clasica = n_comp * 2**n_comp  # FFT clásica: N·log₂N operaciones
    cuantica = n_comp**2          # QFT cuántica: O(n²) puertas
    print(f'  n={n_comp:3d}: FFT clásica={clasica:>15,} ops,  QFT circuito={cuantica:>6} puertas')

## Ejercicio 8.3 — QFT en Qiskit

In [ ]:
def qft_qiskit(n_qubits, inverse=False):
    """Construye el circuito QFT en Qiskit."""
    qc = QuantumCircuit(n_qubits)
    
    def qft_paso(qc, n):
        if n == 0:
            return
        n -= 1
        qc.h(n)  # Hadamard en el qubit más significativo
        for qubit in range(n):
            # Puerta de fase controlada CR_k
            qc.cp(2 * np.pi / 2**(n - qubit + 1), qubit, n)
        qft_paso(qc, n)  # Recursión
    
    qft_paso(qc, n_qubits)
    
    # Intercambiar el orden de los qubits (convención de Qiskit)
    for i in range(n_qubits // 2):
        qc.swap(i, n_qubits - i - 1)
    
    if inverse:
        return qc.inverse()
    return qc

n = 3
qc_qft = qft_qiskit(n)
print(f'Circuito QFT ({n} qubits):')
print(qc_qft.draw(output='text'))
print(f'\nNúmero de puertas: {qc_qft.count_ops()}')

# Verificar contra la matriz teórica
sv_00 = Statevector.from_label('000')
sv_qft = sv_00.evolve(qc_qft)
qft_qiskit_vec = sv_qft.data
qft_teorico = qft_matrix(n) @ np.array([1,0,0,0,0,0,0,0], dtype=complex)

print(f'\nQFT|000⟩ (Qiskit): {np.round(qft_qiskit_vec, 4)}')
print(f'QFT|000⟩ (teoría): {np.round(qft_teorico, 4)}')
print(f'¿Equivalentes? {np.allclose(np.abs(qft_qiskit_vec), np.abs(qft_teorico))}')

## Ejercicio 8.4 — Aplicación: detectar periodicidad
La QFT convierte periodicidad en amplitudes en picos de frecuencia.

In [ ]:
# Estado periódico: amplitudes no nulas cada 'periodo' posiciones
def estado_periodico(N, periodo):
    """Estado con amplitudes iguales en posiciones 0, periodo, 2·periodo..."""
    psi = np.zeros(N, dtype=complex)
    for i in range(0, N, periodo):
        psi[i] = 1.0
    return psi / np.linalg.norm(psi)

N = 16  # n=4 qubits
QFT16 = qft_matrix(4)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
periodos = [1, 2, 4, 8]

for col, periodo in enumerate(periodos):
    psi = estado_periodico(N, periodo)
    qft_psi = QFT16 @ psi
    
    ax_orig = axes[0, col]
    ax_qft  = axes[1, col]
    
    ax_orig.bar(range(N), np.abs(psi)**2, color='steelblue')
    ax_orig.set_title(f'Periodo={periodo}')
    ax_orig.set_xlabel('Estado |j⟩')
    if col == 0: ax_orig.set_ylabel('P(j)')
    
    ax_qft.bar(range(N), np.abs(qft_psi)**2, color='darkorange')
    ax_qft.set_xlabel('Frecuencia k')
    if col == 0: ax_qft.set_ylabel('|QFT(j)|²')

axes[0, 0].set_ylabel('Estado original\nP(j)')
axes[1, 0].set_ylabel('Tras QFT\n|QFT(j)|²')
plt.suptitle('QFT: periodicidad en amplitudes → picos en frecuencia')
plt.tight_layout()
plt.savefig('qft_periodicidad.png', dpi=100, bbox_inches='tight')
plt.show()

print('CLAVE: La QFT mapea un estado con periodo r a picos en múltiplos de N/r.')
print('Esto es exactamente lo que usa el algoritmo de Shor para factorizar.')

## Mini reto — Completar antes de avanzar a la Semana 9

Implementa la QFT inversa (IQFT) y úsala para resolver un problema de estimación de fase simple:
- Dado un estado eigen |u⟩ con eigenvalue e^(2πiφ) de una puerta U
- Usa QPE (Quantum Phase Estimation) básica para estimar φ con 3 bits de precisión
- Usa U = Z (eigenvalues ±1, fases 0 y 1/2) y verifica que obtienes 000 para φ=0 y 100 para φ=1/2

In [ ]:
# TU CÓDIGO AQUÍ
def qpe_basico(n_ancilla: int, U: np.ndarray, eigenstate: np.ndarray) -> dict:
    """Estimación de fase cuántica básica."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 8

Antes de avanzar, comprueba que puedes:

- [ ] Construir la matriz QFT y verificar que es unitaria
- [ ] Comparar QFT con numpy.fft y confirmar equivalencia
- [ ] Implementar el circuito QFT en Qiskit
- [ ] Explicar la complejidad O(n²) vs O(N·log N) clásica
- [ ] Usar la QFT para detectar periodicidad en un estado

## Reflexión (escribe aquí tu respuesta)

**¿Por qué la QFT no puede usarse directamente para acelerar la FFT clásica?**

*(tu respuesta aquí)*

**¿Qué propiedad de la QFT es esencial para el algoritmo de Shor?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 9 — Estimación de Fase y Algoritmo de Shor*